# Cruce universal de dos bases · ProColombia — GIC (Analítica)

Este cuaderno realiza un **cruce (record linkage) bipartito A ↔ B** entre dos bases usando la librería `rues-linker`. Está diseñado para ser **universal y altamente editable**: sirve para cualquier par de bases (Excel o CSV), cualquier hoja o formato, y admite **una, dos o más variables de empalme**, cada una con su propio método de comparación.

---

## Cómo se usa (dos pasos)

1. **Ejecute la _Celda A — EXTRAS_ una sola vez.** Carga toda la lógica del cruce en memoria. No requiere edición.
2. **Edite y ejecute la _Celda B — EJECUTAR_.** Es la **única** celda que usted modifica: allí declara las rutas, las dos bases y los cruces a correr.

Las celdas C y D son opcionales (inspección de resultados y cierre de la sesión).

---

## Prerrequisitos en Google Drive

- Una carpeta con el **paquete `rues-linker` descomprimido** (la que contiene `pyproject.toml`). Es la `RUTA_PAQUETE`.
- Una carpeta con **los dos archivos de datos** (Excel o CSV). Es el `DIR_DATOS`.
- Las salidas se escriben automáticamente en `DIR_DATOS/cruces/<nombre_del_cruce>_<fecha_hora>/`.

---

## Cómo se comparan los nombres (por qué no hay falsos positivos)

El problema clásico: el Jaro-Winkler sobre el nombre completo confunde empresas que comparten una **palabra genérica larga** (CORPORATION, GROUP, INTERNATIONAL, TECHNOLOGIES, INVESTMENTS…) y difieren solo en un token corto y distintivo. Por eso `ACETO CORPORATION` y `AGCO CORPORATION` daban ~0.95, un **falso positivo**.

La solución (comparador **`nombre_idf`**, recomendado) pondera cada token por su **IDF** sobre el corpus de ambas bases: los tokens frecuentes/genéricos pesan casi cero y los raros/distintivos mandan. La similitud de nombre es una mezcla:

- `sim = idf_weight · coseno_IDF(tokens) + (1 − idf_weight) · Jaro-Winkler`

El coseno IDF separa lo que el Jaro-Winkler no puede; el término Jaro-Winkler aporta robustez ante erratas. Con esto, los pares que solo comparten una palabra genérica caen por debajo del umbral y **desaparecen**, mientras las variantes reales (sufijos, país, orden de palabras) se conservan.

---

## Cómo editar la Celda B

- **Rutas:** ajuste `RUTA_PAQUETE` y `DIR_DATOS`.
- **Cada base (`EspecBase`):** indique `archivo`, `hoja` y `fila_encabezado` (índice 0-based de la fila con los encabezados reales; si el Excel tiene títulos o notas arriba, apunte a la fila correcta). En `columnas_extra` liste columnas adicionales que quiera ver en el resultado.
- **Cada cruce (`ConfigCruce`):** lista de variables + umbrales. Puede tener uno, dos o los cruces que necesite; ponga `activo=False` para saltar uno sin borrarlo.
- **Cada variable (`EspecVariable`):** nombre técnico, columna en A, columna en B, `comparador` y `rol`.

---

## Roles de variable

- **`clave`** — genera los candidatos (bloqueo) y participa del puntaje. Debe haber **exactamente una** por cruce. Normalmente es la razón social.
- **`score`** — evidencia adicional que **suma** al puntaje combinado (por ejemplo, ciudad o teléfono).
- **`guardrail`** — **no** entra al puntaje: se compara aparte y, si difiere, **reclasifica el match a “revisión”**; nunca lo descarta en silencio. Rol recomendado para el país.
- **`veto`** — con ambos lados presentes, una discrepancia fuerte **prohíbe** el match (por ejemplo, dos NIT distintos).

---

## Catálogo de comparadores

- **`nombre_idf`** — coseno IDF de tokens mezclado con Jaro-Winkler. **Recomendado para razón social.**
- **`nombre_jaro`** — Jaro-Winkler puro (útil si prefiere no ponderar por IDF).
- **`nombre_token_set` / `nombre_token_sort`** — comparación por conjunto/orden de palabras.
- **`categorico`** — igualdad normalizada de categorías (recomendado para país).
- **`exacto`** — igualdad exacta tras normalizar.
- **`nit`** — igualdad de identificador con dígito de verificación.
- **`telefono`** — coincidencia por los últimos dígitos.
- **`email`** — dominio y parte local del correo.
- **`ciudad`** — igualdad de ciudad normalizada.
- **`direccion`** — coincidencia por tokens de dirección.

Umbrales (`umbral_seguro`, `umbral_revision`, `umbral_concordancia`, `idf_weight`) van en **[0, 1]**. Defaults sensatos: `umbral_seguro=0.90`, `umbral_revision=0.80`, `idf_weight=0.6`. Baje `idf_weight` hacia 0.5 si necesita más recall; súbalo hacia 0.7 para más precisión.

---

## Niveles del resultado

- **`MATCH_SEGURO`** — similitud combinada ≥ `umbral_seguro` (apto para uso automático, con revisión por muestreo).
- **`MATCH_REVISION`** — entre `umbral_revision` y `umbral_seguro` (requiere revisión manual). Aquí caen variantes legítimas: sufijos societarios, país/ciudad añadida, orden de palabras.
- **`MATCH_REVISION_GUARDRAIL`** — era un match seguro pero un guardrail (p. ej., el país) discrepa; se conserva y se marca para revisión.

---

## Advertencia de calidad

Sin un conjunto de verdad etiquetado no es posible garantizar F1; este cuaderno **prioriza la precisión y la auditabilidad sobre el recall**. Revise **siempre** el nivel de _revisión_ antes de dar un cruce por definitivo. Principio editorial: *ante la duda, excluir; ante la contradicción, explicar; ante la ambigüedad, advertir.*


## Instalación de dependencias de salida  ·  *ejecutar una vez*

Instala `xlsxwriter` (Excel con formato), `openpyxl` y `pyarrow` (Parquet).

In [1]:
# Dependencias de salida (una vez). xlsxwriter escribe el Excel con formato;
# openpyxl lo lee; pyarrow habilita Parquet. (La Celda A también las asegura.)
!pip install -q xlsxwriter openpyxl pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.2 MB/s eta 0:00:00


## Celda A — EXTRAS  ·  *ejecutar una sola vez*

Carga toda la lógica del cruce (v2, con puntuación por IDF) y las utilidades de Colab (montaje de Drive e instalación del paquete). **No requiere edición.**

In [2]:
"""Cruce universal de dos bases — record linkage bipartito A ↔ B.

Contexto: Google Colab Free. I/O sobre Google Drive. Entrada: dos archivos
Excel/CSV con una o más variables de empalme. Usa la librería `rues-linker`
para la LIMPIEZA de nombres y el BLOQUEO (generación de candidatos), y un
motor de puntuación PROPIO, transparente y vectorizado, para el scoring.

Por qué un scoring propio (v2, 2026-07-09):
    El Jaro-Winkler sobre el nombre completo produce FALSOS POSITIVOS cuando dos
    empresas comparten una palabra genérica larga (CORPORATION, INTERNATIONAL,
    GROUP, TECHNOLOGIES, INVESTMENTS…) y difieren solo en un token corto y
    distintivo (p. ej. ``ACETO CORPORATION`` vs ``AGCO CORPORATION`` ≈ 0.95). La
    palabra genérica domina la cadena e infla la similitud.

    Solución (la misma que documenta la propia librería para fuentes sin NIT):
    RE-PESADO POR IDF DE TOKENS. Cada token se pondera por su IDF sobre el corpus
    (ambas bases): los tokens frecuentes/genéricos pesan casi cero; los raros y
    distintivos mandan. La similitud de nombre es una MEZCLA:

        sim_nombre = idf_weight · coseno_IDF(tokens)  +  (1 − idf_weight) · JaroWinkler

    El coseno IDF separa lo que el JW no puede (medido sobre datos reales:
    falsos positivos ≤ 0.60, verdaderos = 1.00), y el término JW aporta robustez
    ante erratas y diferencias de segmentación de tokens. Todo vectorizado
    (matriz dispersa token×registro + producto punto por fila).

Diseño de variables:
    Cada variable de unión declara comparador, umbral y ROL:
        - clave     : genera candidatos (bloqueo) y puntúa. Exactamente una.
        - score     : suma evidencia al puntaje combinado.
        - guardrail : NO puntúa; reclasifica a revisión si difiere (nunca descarta).
        - veto      : una discrepancia fuerte (ambos lados presentes) prohíbe el match.

Author: GIC-ProColombia (Analítica)  ·  Date: 2026-07-09  ·  Version: 2.0.0
"""

from __future__ import annotations

import json
import math
import time
import unicodedata
from collections.abc import Callable
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# NOTA: los símbolos de `record_linkage` y `scipy` se importan de forma PEREZOSA
# dentro de cada función (así la celda EXTRAS se define sin el paquete instalado;
# la celda EJECUTAR lo instala y luego corre).

# ═══════════════════════════════════════════════════════════════════════════
# ENTORNO COLAB — montaje de Drive e instalación del paquete (ejecutar una vez)
# ═══════════════════════════════════════════════════════════════════════════
def montar_drive_con_retry(intentos: int = 3, espera: int = 5) -> None:
    """Monta Google Drive con reintentos. Idempotente (no re-monta si ya está)."""
    if Path("/content/drive/MyDrive").exists():
        print("✅ Google Drive ya está montado.")
        return
    from google.colab import drive  # solo existe dentro de Colab

    ultimo: Exception | None = None
    for i in range(1, intentos + 1):
        try:
            drive.mount("/content/drive", force_remount=False)
            print("✅ Google Drive montado.")
            return
        except Exception as exc:  # noqa: BLE001
            ultimo = exc
            print(f"⚠️  Intento {i}/{intentos} de montar Drive falló: {exc}")
            time.sleep(espera)
    raise RuntimeError(f"No se pudo montar Google Drive tras {intentos} intentos: {ultimo}")


def instalar_paquete(ruta_paquete: str, version_esperada: str = "3.2.3") -> None:
    """Deja disponible `record_linkage` desde una ruta local en Drive.

    Estrategia en cascada, robusta ante los reinicios de entorno de Colab:
        1) pip install -e (editable) desde la ruta del paquete;
        2) si el import sigue fallando, pip install --force-reinstall --no-deps;
        3) como último recurso, agrega la ruta a sys.path.
    Además asegura dependencias de runtime que el paquete no declara (pytz, h5py).
    La ``ruta_paquete`` debe contener el proyecto descomprimido (con pyproject.toml).
    """
    import importlib
    import subprocess
    import sys

    ruta = Path(ruta_paquete)
    if not ruta.exists():
        raise FileNotFoundError(
            f"No existe la RUTA_PAQUETE: {ruta}. Verifique que descomprimió el .zip "
            f"de rues-linker en esa carpeta de Drive."
        )

    def _pip(*args: str) -> None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

    for dep in ("pytz", "h5py", "xlsxwriter", "openpyxl", "pyarrow"):
        try:
            importlib.import_module(dep)
        except ImportError:
            _pip(dep)

    def _disponible() -> bool:
        try:
            import record_linkage  # noqa: F401

            print(f"✅ record_linkage disponible (versión {getattr(record_linkage, '__version__', '?')}).")
            return True
        except Exception:  # noqa: BLE001
            return False

    if _disponible():
        return
    print("⏳ Instalando rues-linker en modo editable…")
    _pip("-e", str(ruta))
    importlib.invalidate_caches()
    if _disponible():
        return
    print("⏳ Reintentando con --force-reinstall…")
    _pip("--force-reinstall", "--no-deps", str(ruta))
    importlib.invalidate_caches()
    if _disponible():
        return
    print("⏳ Último recurso: agregando la ruta del paquete a sys.path…")
    if str(ruta) not in sys.path:
        sys.path.insert(0, str(ruta))
    importlib.invalidate_caches()
    if not _disponible():
        raise RuntimeError(
            "No fue posible cargar record_linkage. Revise que la RUTA_PAQUETE apunte "
            "a la carpeta con el proyecto descomprimido (debe contener pyproject.toml)."
        )



__version__ = "2.0.0"

# ═══════════════════════════════════════════════════════════════════════════
# 1. CATÁLOGO DE COMPARADORES
# ═══════════════════════════════════════════════════════════════════════════
COMPARADORES_VALIDOS: frozenset[str] = frozenset(
    {
        "nombre_idf",
        "nombre_jaro",
        "nombre_token_set",
        "nombre_token_sort",
        "categorico",
        "exacto",
        "nit",
        "telefono",
        "email",
        "ciudad",
        "direccion",
    }
)
COMPARADORES_DE_NOMBRE: frozenset[str] = frozenset(
    {"nombre_idf", "nombre_jaro", "nombre_token_set", "nombre_token_sort"}
)
COMPARADORES_CATEGORICOS: frozenset[str] = frozenset({"categorico", "exacto", "ciudad"})
ROLES_VALIDOS: frozenset[str] = frozenset({"clave", "score", "guardrail", "veto"})


def _crear_comparador(clave: str) -> Any:
    """Crea la instancia de un comparador de `record_linkage` (import perezoso).

    No cubre 'nombre_idf', que se computa aparte en `_similitud_variable`.
    """
    from record_linkage.matching import (
        AddressTokenSet,
        CategoricalSigned,
        CityNormalizedEqual,
        EmailDomainLocal,
        ExactSigned,
        ExactWithDV,
        JaroWinklerSigned,
        PhoneLastDigits,
        TokenSetSigned,
        TokenSortSigned,
    )

    fabricas: dict[str, Callable[[], Any]] = {
        "nombre_jaro": lambda: JaroWinklerSigned(prefix_weight=0.1),
        "nombre_token_set": TokenSetSigned,
        "nombre_token_sort": TokenSortSigned,
        "categorico": CategoricalSigned,
        "exacto": ExactSigned,
        "nit": ExactWithDV,
        "telefono": lambda: PhoneLastDigits(n=7),
        "email": EmailDomainLocal,
        "ciudad": CityNormalizedEqual,
        "direccion": AddressTokenSet,
    }
    return fabricas[clave]()


_PRIORIDAD_TIER: dict[str, int] = {
    "MATCH_SEGURO": 1,
    "MATCH_REVISION": 2,
    "MATCH_REVISION_GUARDRAIL": 3,
    "NO_MATCH": 9,
}

CANON_PAISES_DEFAULT: dict[str, str] = {
    "ESTADOS UNIDOS DE AMERICA": "ESTADOS UNIDOS",
    "EE UU": "ESTADOS UNIDOS",
    "EEUU": "ESTADOS UNIDOS",
    "USA": "ESTADOS UNIDOS",
    "UNITED STATES": "ESTADOS UNIDOS",
    "(EN BLANCO)": "",
    "(BLANK)": "",
    "SIN INFORMACION": "",
    "N/A": "",
    "NA": "",
}


# ═══════════════════════════════════════════════════════════════════════════
# 2. NORMALIZACIÓN DE TEXTO (vectorizada)
# ═══════════════════════════════════════════════════════════════════════════
def sin_acentos_upper(serie: pd.Series) -> pd.Series:
    """Normaliza una serie a ASCII-mayúsculas sin acentos, recortada."""
    s = serie.astype("string").fillna("").str.strip().str.upper()

    def _strip(x: str) -> str:
        return "".join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))

    return s.map(_strip)


def canonizar(serie: pd.Series, mapa: dict[str, str]) -> pd.Series:
    """Normaliza y aplica un mapa canónico (sinónimos → forma única)."""
    base = sin_acentos_upper(serie)
    if mapa:
        base = base.replace(mapa)
    return base


def _a_similitud(signed: np.ndarray) -> np.ndarray:
    """Convierte score firmado [-1, +1] a similitud [0, 1]."""
    return (np.asarray(signed, dtype=np.float64) + 1.0) / 2.0


# ═══════════════════════════════════════════════════════════════════════════
# 3. CONFIGURACIÓN (dataclasses con validación)
# ═══════════════════════════════════════════════════════════════════════════
@dataclass(frozen=True)
class EspecVariable:
    """Una variable de unión: qué columnas comparar, cómo y con qué rol.

    Args:
        nombre: Nombre técnico en el combinado (p. ej. ``"RAZON_SOCIAL"``).
        col_a: Columna en la Base A.
        col_b: Columna en la Base B.
        comparador: Clave de ``COMPARADORES_VALIDOS``. Recomendado ``"nombre_idf"``
            para razón social (robusto a palabras genéricas compartidas).
        rol: ``"clave"`` | ``"score"`` | ``"guardrail"`` | ``"veto"``.
        umbral_concordancia: Similitud mínima [0, 1] para "concordar". Default 0.80.
        peso: Peso relativo en el score combinado. Default 1.0.
    """

    nombre: str
    col_a: str
    col_b: str
    comparador: str = "nombre_idf"
    rol: str = "score"
    umbral_concordancia: float = 0.80
    peso: float = 1.0

    def __post_init__(self) -> None:
        if self.comparador not in COMPARADORES_VALIDOS:
            raise ValueError(
                f"[{self.nombre}] comparador '{self.comparador}' desconocido. "
                f"Opciones: {sorted(COMPARADORES_VALIDOS)}"
            )
        if self.rol not in ROLES_VALIDOS:
            raise ValueError(
                f"[{self.nombre}] rol '{self.rol}' inválido. Opciones: {sorted(ROLES_VALIDOS)}"
            )
        if not 0.0 <= self.umbral_concordancia <= 1.0:
            raise ValueError(f"[{self.nombre}] umbral_concordancia ∈ [0,1].")
        if self.peso < 0:
            raise ValueError(f"[{self.nombre}] peso debe ser >= 0.")


@dataclass(frozen=True)
class EspecBase:
    """Descripción de una base de entrada (archivo, hoja, encabezado)."""

    nombre: str
    archivo: str
    hoja: str = "Sheet1"
    fila_encabezado: int = 0
    columnas_extra: tuple[str, ...] = ()
    prefijo_id: str = "R"

    def __post_init__(self) -> None:
        if self.fila_encabezado < 0:
            raise ValueError(f"[{self.nombre}] fila_encabezado debe ser >= 0")


@dataclass(frozen=True)
class ConfigCruce:
    """Configuración completa de UN cruce entre dos bases.

    Args:
        nombre: Etiqueta del cruce (p. ej. ``"01_solo_nombre"``).
        variables: Lista de ``EspecVariable``. Exactamente una con rol ``"clave"``.
        umbral_seguro: Similitud combinada [0, 1] desde la que un match es SEGURO.
            Default 0.90.
        umbral_revision: Similitud combinada mínima [0, 1] para REPORTAR un match;
            por debajo se descarta. Entre revisión y seguro → REVISIÓN. Default 0.80.
        idf_weight: Peso del coseno IDF en la similitud de nombre (0..1). El resto
            lo aporta Jaro-Winkler. 0.6 discrimina bien y tolera erratas. Default 0.6.
        min_concordancias: Nº mínimo de variables puntuables que deben "concordar".
            Default 1.
        veto_umbral_sim: Similitud por debajo de la cual una variable con rol
            ``"veto"`` (con ambos lados presentes) prohíbe el match. Default 0.5.
        modo_limpieza_nombres: 'CONSERVADOR' | 'BALANCEADO' | 'AGRESIVO'. Default
            'BALANCEADO'.
        max_bucket_size: Cota del tamaño de bucket en el bloqueo. Default 400.
        usar_pass_fingerprint / usar_pass_token: pasadas de bloqueo. Default True.
        activo: Si False, el cruce se salta. Default True.
    """

    nombre: str
    variables: tuple[EspecVariable, ...]
    umbral_seguro: float = 0.90
    umbral_revision: float = 0.80
    idf_weight: float = 0.6
    min_concordancias: int = 1
    veto_umbral_sim: float = 0.5
    modo_limpieza_nombres: str = "BALANCEADO"
    max_bucket_size: int = 400
    usar_pass_fingerprint: bool = True
    usar_pass_token: bool = True
    activo: bool = True

    def __post_init__(self) -> None:
        if not self.variables:
            raise ValueError(f"[{self.nombre}] requiere al menos una variable.")
        claves = [v for v in self.variables if v.rol == "clave"]
        if len(claves) != 1:
            raise ValueError(
                f"[{self.nombre}] debe declarar EXACTAMENTE una variable con rol='clave'. "
                f"Encontradas: {[v.nombre for v in claves]}"
            )
        nombres = [v.nombre for v in self.variables]
        if len(nombres) != len(set(nombres)):
            raise ValueError(f"[{self.nombre}] nombres de variable duplicados: {nombres}")
        if not 0.0 <= self.umbral_revision <= self.umbral_seguro <= 1.0:
            raise ValueError(
                f"[{self.nombre}] se requiere 0 <= umbral_revision ({self.umbral_revision}) "
                f"<= umbral_seguro ({self.umbral_seguro}) <= 1."
            )
        if not 0.0 <= self.idf_weight <= 1.0:
            raise ValueError(f"[{self.nombre}] idf_weight ∈ [0,1], recibido {self.idf_weight}.")
        if self.min_concordancias < 1:
            raise ValueError(f"[{self.nombre}] min_concordancias debe ser >= 1.")
        if self.modo_limpieza_nombres not in {"CONSERVADOR", "BALANCEADO", "AGRESIVO"}:
            raise ValueError(f"[{self.nombre}] modo_limpieza_nombres inválido.")
        if self.max_bucket_size < 2:
            raise ValueError(f"[{self.nombre}] max_bucket_size debe ser >= 2")

    @property
    def clave(self) -> EspecVariable:
        """La variable de bloqueo (rol='clave')."""
        return next(v for v in self.variables if v.rol == "clave")

    @property
    def variables_puntuables(self) -> list[EspecVariable]:
        """Variables que suman al score y a las concordancias (clave, score)."""
        return [v for v in self.variables if v.rol in {"clave", "score"}]

    @property
    def variables_veto(self) -> list[EspecVariable]:
        """Variables con rol veto."""
        return [v for v in self.variables if v.rol == "veto"]

    @property
    def variables_guardrail(self) -> list[EspecVariable]:
        """Variables con rol guardrail (reclasifican, no descartan)."""
        return [v for v in self.variables if v.rol == "guardrail"]

    @property
    def variables_con_similitud(self) -> list[EspecVariable]:
        """Todas las variables que requieren cálculo de similitud (no guardrail)."""
        return [v for v in self.variables if v.rol in {"clave", "score", "veto"}]


@dataclass(frozen=True)
class RutasCruce:
    """Rutas de un cruce (fuente única de verdad, vía pathlib)."""

    dir_datos: Path
    archivo_a: Path
    archivo_b: Path
    dir_salida: Path

    def crear_directorios(self) -> None:
        """Crea el directorio de salida (idempotente)."""
        self.dir_salida.mkdir(parents=True, exist_ok=True)


def construir_rutas(
    dir_datos: str | Path,
    espec_a: EspecBase,
    espec_b: EspecBase,
    nombre_cruce: str,
    timestamp: str | None = None,
) -> RutasCruce:
    """Construye las rutas de un cruce a partir del directorio de datos."""
    dir_datos = Path(dir_datos)
    ts = timestamp or datetime.now().strftime("%Y%m%d_%H%M%S")
    return RutasCruce(
        dir_datos=dir_datos,
        archivo_a=dir_datos / espec_a.archivo,
        archivo_b=dir_datos / espec_b.archivo,
        dir_salida=dir_datos / "cruces" / f"{nombre_cruce}_{ts}",
    )


# ═══════════════════════════════════════════════════════════════════════════
# 4. LECTURA / PRE-FLIGHT (validar TODO antes de tocar datos)
# ═══════════════════════════════════════════════════════════════════════════
def _leer_excel_o_csv(ruta: Path, hoja: str, fila_encabezado: int) -> pd.DataFrame:
    """Lee un archivo Excel/CSV/Parquet como texto (dtype=str) con encabezado dado."""
    suf = ruta.suffix.lower()
    if suf in {".xlsx", ".xls", ".xlsm"}:
        df = pd.read_excel(ruta, sheet_name=hoja, header=fila_encabezado, dtype=str)
    elif suf in {".csv", ".txt"}:
        df = pd.read_csv(ruta, header=fila_encabezado, dtype=str, sep=None, engine="python")
    elif suf == ".parquet":
        df = pd.read_parquet(ruta).astype("string")
    else:
        raise ValueError(f"Extensión no soportada: {suf} ({ruta.name})")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def preflight(
    espec_a: EspecBase,
    espec_b: EspecBase,
    rutas: RutasCruce,
    cruces: dict[str, ConfigCruce],
) -> None:
    """Valida archivos, hojas y columnas de todas las variables de cruces activos.

    Lanza un único ``RuntimeError`` con TODOS los problemas encontrados.
    """
    errores: list[str] = []
    for espec, ruta in [(espec_a, rutas.archivo_a), (espec_b, rutas.archivo_b)]:
        if not ruta.exists():
            errores.append(f"No existe el archivo de {espec.nombre}: {ruta}")
    if errores:
        raise RuntimeError("❌ Pre-flight falló:\n" + "\n".join(f"  • {e}" for e in errores))

    cols: dict[str, set[str]] = {}
    for espec, ruta in [(espec_a, rutas.archivo_a), (espec_b, rutas.archivo_b)]:
        try:
            df = _leer_excel_o_csv(ruta, espec.hoja, espec.fila_encabezado)
            cols[espec.nombre] = set(df.columns)
        except Exception as exc:
            errores.append(
                f"No pude leer {espec.nombre} (hoja='{espec.hoja}', "
                f"fila_encabezado={espec.fila_encabezado}): {type(exc).__name__}: {exc}"
            )
    if errores:
        raise RuntimeError("❌ Pre-flight falló:\n" + "\n".join(f"  • {e}" for e in errores))

    activos = {n: c for n, c in cruces.items() if c.activo}
    if not activos:
        raise RuntimeError("❌ Pre-flight: no hay cruces activos (activo=True).")

    for nombre, cfg in activos.items():
        for v in cfg.variables:
            if v.col_a not in cols[espec_a.nombre]:
                errores.append(
                    f"[{nombre}/{v.nombre}] columna '{v.col_a}' no está en {espec_a.nombre}. "
                    f"Muestra: {sorted(cols[espec_a.nombre])[:8]}…"
                )
            if v.col_b not in cols[espec_b.nombre]:
                errores.append(
                    f"[{nombre}/{v.nombre}] columna '{v.col_b}' no está en {espec_b.nombre}. "
                    f"Muestra: {sorted(cols[espec_b.nombre])[:8]}…"
                )
    for espec in (espec_a, espec_b):
        for col in espec.columnas_extra:
            if col not in cols[espec.nombre]:
                errores.append(f"[{espec.nombre}] columna_extra '{col}' no existe.")

    if errores:
        raise RuntimeError("❌ Pre-flight falló:\n" + "\n".join(f"  • {e}" for e in errores))
    print(f"✅ Pre-flight OK — {len(activos)} cruce(s) activo(s); columnas verificadas.")


# ═══════════════════════════════════════════════════════════════════════════
# 5. CARGA Y PREPARACIÓN DEL COMBINADO
# ═══════════════════════════════════════════════════════════════════════════
def cargar_base(espec: EspecBase, ruta: Path) -> pd.DataFrame:
    """Carga una base y le asigna un ID de fila estable (p. ej. ``A_000001``)."""
    df = _leer_excel_o_csv(ruta, espec.hoja, espec.fila_encabezado).reset_index(drop=True)
    df["_ID"] = [f"{espec.prefijo_id}_{i:06d}" for i in range(len(df))]
    print(f"✅ {espec.nombre}: {len(df):,} filas × {df.shape[1] - 1} columnas — {ruta.name}")
    return df


def preparar_combinado(
    df_a: pd.DataFrame,
    df_b: pd.DataFrame,
    espec_a: EspecBase,
    espec_b: EspecBase,
    config: ConfigCruce,
    canon_categoricos: dict[str, str],
) -> tuple[pd.DataFrame, dict[str, int]]:
    """Construye el combinado con columnas técnicas (``<var>`` y ``<var>_RAW``).

    RangeIndex contiguo (requisito del bloqueo y del lookup posicional del scorer),
    más ``_source``, ``_id`` y las columnas extra de cada base.
    """
    from record_linkage.processing.text import TextProcessor

    clave = config.clave
    tp = TextProcessor(cleaning_mode=config.modo_limpieza_nombres)

    def _lado(df: pd.DataFrame, espec: EspecBase) -> pd.DataFrame:
        es_a = espec.nombre == espec_a.nombre
        out = pd.DataFrame(index=df.index)
        out["_source"] = espec.nombre
        out["_id"] = df["_ID"].to_numpy()
        for v in config.variables:
            col = v.col_a if es_a else v.col_b
            raw = df[col].astype("string")
            out[f"{v.nombre}_RAW"] = raw.to_numpy()
            if v.comparador in COMPARADORES_DE_NOMBRE:
                out[v.nombre] = tp.process_series(raw, column_name=v.nombre).to_numpy()
            elif v.comparador in COMPARADORES_CATEGORICOS:
                out[v.nombre] = canonizar(raw, canon_categoricos).to_numpy()
            else:
                out[v.nombre] = raw.fillna("").str.strip().to_numpy()
        for col in espec.columnas_extra:
            out[f"EXTRA_{col}"] = df[col].astype("string").to_numpy()
        return out

    comb = pd.concat([_lado(df_a, espec_a), _lado(df_b, espec_b)], ignore_index=True, sort=False)
    n_antes = len(comb)
    vacia = comb[clave.nombre].astype(str).str.strip().eq("")
    comb = comb.loc[~vacia].reset_index(drop=True)
    meta = {
        "filas_a": int((comb["_source"] == espec_a.nombre).sum()),
        "filas_b": int((comb["_source"] == espec_b.nombre).sum()),
        "descartadas_clave_vacia": int(n_antes - len(comb)),
    }
    print(
        f"✅ Combinado: {len(comb):,} filas (A={meta['filas_a']:,} · B={meta['filas_b']:,}) — "
        f"{meta['descartadas_clave_vacia']:,} descartadas por '{clave.nombre}' vacío."
    )
    return comb, meta


# ═══════════════════════════════════════════════════════════════════════════
# 6. BLOQUEO (candidatos) — vectorizado, solo pares cruzados A↔B
# ═══════════════════════════════════════════════════════════════════════════
def generar_pares_cruzados(comb: pd.DataFrame, config: ConfigCruce, fuente_a: str) -> pd.DataFrame:
    """Genera pares candidatos A↔B por bloqueo de nombre (vectorizado).

    Usa ``block_by_name_multipass`` sobre la variable clave, conserva solo los
    pares que cruzan las dos fuentes y los orienta con A a la izquierda.
    """
    from record_linkage.engine.lsh.name_blocking import (
        NameBlockingConfig,
        block_by_name_multipass,
    )

    cfg_block = NameBlockingConfig(
        enable_fingerprint=config.usar_pass_fingerprint,
        enable_significant_token=config.usar_pass_token,
        max_bucket_size=config.max_bucket_size,
    )
    pares = block_by_name_multipass(comb, name_column=config.clave.nombre, config=cfg_block)
    if not pares:
        print("⚠️  El bloqueo no produjo candidatos.")
        return pd.DataFrame(
            {"id_left": np.array([], dtype=np.int64), "id_right": np.array([], dtype=np.int64)}
        )

    arr = np.fromiter((x for par in pares for x in par), dtype=np.int64).reshape(-1, 2)
    src = comb["_source"].to_numpy()
    arr = arr[src[arr[:, 0]] != src[arr[:, 1]]]
    es_a_izq = src[arr[:, 0]] == fuente_a
    id_left = np.where(es_a_izq, arr[:, 0], arr[:, 1])
    id_right = np.where(es_a_izq, arr[:, 1], arr[:, 0])
    pares_df = pd.DataFrame({"id_left": id_left, "id_right": id_right}).drop_duplicates()
    print(
        f"✅ Candidatos: {len(pares):,} pares por bloqueo → "
        f"{len(pares_df):,} pares cruzados A↔B (únicos)."
    )
    return pares_df.reset_index(drop=True)


# ═══════════════════════════════════════════════════════════════════════════
# 7. SCORING PROPIO (IDF + Jaro-Winkler) + NIVELES + VETO + GUARDRAILS
# ═══════════════════════════════════════════════════════════════════════════
def _matriz_idf(serie_nombre: pd.Series) -> Any:
    """Matriz dispersa token×registro ponderada por IDF y L2-normalizada por fila.

    IDF suavizado: ``idf(t) = ln((N+1)/(df(t)+1)) + 1``. Los tokens frecuentes
    (genéricos) reciben peso bajo; los raros (distintivos), peso alto.
    """
    from scipy import sparse

    textos = serie_nombre.fillna("").astype(str).to_numpy()
    tok_lists = [t.split() for t in textos]
    n = len(textos)
    dfc: dict[str, int] = {}
    for tl in tok_lists:
        for t in set(tl):
            dfc[t] = dfc.get(t, 0) + 1
    idf = {t: math.log((n + 1) / (c + 1)) + 1.0 for t, c in dfc.items()}

    vocab: dict[str, int] = {}
    filas: list[int] = []
    cols: list[int] = []
    datos: list[float] = []
    for i, tl in enumerate(tok_lists):
        for t in set(tl):
            j = vocab.setdefault(t, len(vocab))
            filas.append(i)
            cols.append(j)
            datos.append(idf[t])
    if not vocab:
        return sparse.csr_matrix((n, 1), dtype=np.float64)
    matriz = sparse.csr_matrix((datos, (filas, cols)), shape=(n, len(vocab)), dtype=np.float64)
    normas = np.sqrt(matriz.multiply(matriz).sum(axis=1)).A.ravel()
    normas[normas == 0] = 1.0
    return sparse.diags(1.0 / normas) @ matriz


def _idf_coseno(matriz_norm: Any, il: np.ndarray, ir: np.ndarray) -> np.ndarray:
    """Coseno IDF por par (producto punto de filas L2-normalizadas). Vectorizado."""
    if matriz_norm.shape[1] == 0 or len(il) == 0:
        return np.zeros(len(il), dtype=np.float64)
    return np.asarray(matriz_norm[il].multiply(matriz_norm[ir]).sum(axis=1)).ravel()


def _similitud_variable(
    comb: pd.DataFrame, il: np.ndarray, ir: np.ndarray, var: EspecVariable, idf_weight: float
) -> np.ndarray:
    """Similitud [0,1] de una variable sobre los pares (il, ir).

    ``nombre_idf`` = idf_weight·coseno_IDF + (1−idf_weight)·JaroWinkler. El resto
    usa directamente el comparador vectorizado de la librería.
    """
    izq = comb[var.nombre].to_numpy()[il]
    der = comb[var.nombre].to_numpy()[ir]
    if var.comparador == "nombre_idf":
        matriz = _matriz_idf(comb[var.nombre])
        cos = _idf_coseno(matriz, il, ir)
        jw = _a_similitud(_crear_comparador("nombre_jaro").compare(izq, der))
        return idf_weight * cos + (1.0 - idf_weight) * jw
    comp = _crear_comparador(var.comparador)
    return _a_similitud(comp.compare(izq, der))


def _guardrail_compatibilidad(
    comb: pd.DataFrame, il: np.ndarray, ir: np.ndarray, nombre_var: str
) -> np.ndarray:
    """Compatibilidad de un categórico entre lados. Array objeto True/False/None.

    None cuando falta el valor en algún lado (no informativo); True si coinciden
    tras normalización, False si difieren.
    """
    izq = pd.Series(comb[nombre_var].to_numpy()[il]).fillna("").astype(str).str.strip()
    der = pd.Series(comb[nombre_var].to_numpy()[ir]).fillna("").astype(str).str.strip()
    ambos = ((izq != "") & (der != "")).to_numpy()
    iguales = izq.to_numpy() == der.to_numpy()
    out = np.full(len(il), None, dtype=object)
    out[ambos] = iguales[ambos]
    return out


def puntuar_y_clasificar(
    comb: pd.DataFrame, pares: pd.DataFrame, config: ConfigCruce
) -> pd.DataFrame:
    """Puntúa los pares candidatos y los clasifica en niveles.

    Score combinado = media ponderada (por peso) de las similitudes de las
    variables puntuables (clave, score). Un par es MATCH si no está vetado, su
    score ≥ umbral_revision y tiene ≥ min_concordancias. Un veto (variable veto
    con ambos lados presentes y similitud < veto_umbral_sim) descarta el par. Un
    guardrail discrepante degrada un SEGURO a REVISIÓN sin descartarlo.
    """
    il = pares["id_left"].to_numpy()
    ir = pares["id_right"].to_numpy()
    if len(il) == 0:
        return pd.DataFrame()

    # Similitud por variable (puntuables + veto).
    sims: dict[str, np.ndarray] = {
        v.nombre: _similitud_variable(comb, il, ir, v, config.idf_weight)
        for v in config.variables_con_similitud
    }

    # Score combinado (media ponderada de variables puntuables).
    punt = config.variables_puntuables
    pesos = np.array([v.peso for v in punt], dtype=np.float64)
    pesos = pesos / pesos.sum() if pesos.sum() > 0 else np.ones(len(punt)) / max(len(punt), 1)
    stack = np.vstack([sims[v.nombre] for v in punt])
    score = (pesos[:, None] * stack).sum(axis=0)

    # Concordancias (variables puntuables cuya similitud ≥ su umbral).
    conc = np.zeros(len(il), dtype=np.int64)
    for i, v in enumerate(punt):
        conc += (stack[i] >= v.umbral_concordancia).astype(np.int64)

    # Veto (variables veto con ambos lados presentes y similitud baja).
    vetado = np.zeros(len(il), dtype=bool)
    for v in config.variables_veto:
        izq = pd.Series(comb[v.nombre].to_numpy()[il]).fillna("").astype(str).str.strip().to_numpy()
        der = pd.Series(comb[v.nombre].to_numpy()[ir]).fillna("").astype(str).str.strip().to_numpy()
        ambos = (izq != "") & (der != "")
        vetado |= ambos & (sims[v.nombre] < config.veto_umbral_sim)

    es_match = (~vetado) & (score >= config.umbral_revision) & (conc >= config.min_concordancias)
    idx = np.where(es_match)[0]
    if len(idx) == 0:
        print("⚠️  Ningún par superó los criterios de match.")
        return pd.DataFrame()

    m = pd.DataFrame({"id_left": il[idx], "id_right": ir[idx]})
    m["SCORE_SIM"] = np.round(score[idx], 4)
    m["SCORE_100"] = np.round(score[idx] * 100.0, 2)
    m["concordancias"] = conc[idx]
    m["TIER"] = np.where(score[idx] >= config.umbral_seguro, "MATCH_SEGURO", "MATCH_REVISION")
    for v in config.variables_con_similitud:
        m[f"sim_{v.nombre}"] = np.round(sims[v.nombre][idx], 4)

    # Guardrails: degradan SEGURO → REVISION_GUARDRAIL si discrepan (no descartan).
    flag = np.array([""] * len(idx), dtype=object)
    for v in config.variables_guardrail:
        compat = _guardrail_compatibilidad(comb, il[idx], ir[idx], v.nombre)
        m[f"{v.nombre}_COMPATIBLE"] = compat
        discrepa = np.array([c is False for c in compat])
        tier = m["TIER"].to_numpy()
        tier[discrepa & (tier == "MATCH_SEGURO")] = "MATCH_REVISION_GUARDRAIL"
        m["TIER"] = tier
        flag = np.where(discrepa, flag + f"{v.nombre}_DISTINTO;", flag)
    if config.variables_guardrail:
        m["FLAG_GUARDRAIL"] = flag

    m["_PRIORIDAD"] = m["TIER"].map(_PRIORIDAD_TIER).fillna(9).astype(int)
    print(f"✅ Matches reportables: {len(m):,} — composición: {m['TIER'].value_counts().to_dict()}")
    return m


# ═══════════════════════════════════════════════════════════════════════════
# 8. ENRIQUECIMIENTO Y SALIDAS
# ═══════════════════════════════════════════════════════════════════════════
def construir_correlativa(
    matches: pd.DataFrame, comb: pd.DataFrame, config: ConfigCruce
) -> pd.DataFrame:
    """Arma la correlativa legible A↔B (una fila por par emparejado)."""
    if matches.empty:
        return pd.DataFrame()
    il = matches["id_left"].to_numpy()
    ir = matches["id_right"].to_numpy()

    datos: dict[str, Any] = {
        "ID_A": comb["_id"].to_numpy()[il],
        "ID_B": comb["_id"].to_numpy()[ir],
        "TIER": matches["TIER"].to_numpy(),
        "SCORE_SIM": matches["SCORE_SIM"].to_numpy(),
        "SCORE_100": matches["SCORE_100"].to_numpy(),
        "concordancias": matches["concordancias"].to_numpy(),
    }
    for v in config.variables:
        datos[f"{v.nombre}_A"] = comb[f"{v.nombre}_RAW"].to_numpy()[il]
        datos[f"{v.nombre}_B"] = comb[f"{v.nombre}_RAW"].to_numpy()[ir]
        if f"sim_{v.nombre}" in matches.columns:
            datos[f"sim_{v.nombre}"] = matches[f"sim_{v.nombre}"].to_numpy()
        if f"{v.nombre}_COMPATIBLE" in matches.columns:
            datos[f"{v.nombre}_COMPATIBLE"] = matches[f"{v.nombre}_COMPATIBLE"].to_numpy()
    if "FLAG_GUARDRAIL" in matches.columns:
        datos["FLAG_GUARDRAIL"] = matches["FLAG_GUARDRAIL"].to_numpy()
    datos[f"{config.clave.nombre}_A_LIMPIO"] = comb[config.clave.nombre].to_numpy()[il]
    datos[f"{config.clave.nombre}_B_LIMPIO"] = comb[config.clave.nombre].to_numpy()[ir]
    for col in [c for c in comb.columns if c.startswith("EXTRA_")]:
        datos[f"{col[6:]}_A"] = comb[col].to_numpy()[il]
        datos[f"{col[6:]}_B"] = comb[col].to_numpy()[ir]

    corr = pd.DataFrame(datos)
    corr["_prio"] = matches["_PRIORIDAD"].to_numpy()
    corr = corr.sort_values(["_prio", "SCORE_SIM"], ascending=[True, False]).drop(columns="_prio")
    return corr.reset_index(drop=True)


def construir_base_a_con_match(
    df_a: pd.DataFrame, correlativa: pd.DataFrame, config: ConfigCruce
) -> pd.DataFrame:
    """Anota la Base A con su MEJOR match en B (una fila por empresa A).

    'Mejor' = mayor prioridad de nivel, luego mayor SCORE_SIM. Las empresas sin
    match quedan con ``TIER_MATCH='NO_MATCH'``.
    """
    base = df_a.copy()
    if correlativa.empty:
        base["TIENE_MATCH"] = False
        base["TIER_MATCH"] = "NO_MATCH"
        return base

    corr = correlativa.assign(_p=correlativa["TIER"].map(_PRIORIDAD_TIER).fillna(9))
    mejor = (
        corr.sort_values(["_p", "SCORE_SIM"], ascending=[True, False])
        .drop_duplicates("ID_A", keep="first")
        .copy()
    )
    col_nombre_b = f"{config.clave.nombre}_B"
    ren = {
        "ID_B": "ID_B_MATCH",
        "TIER": "TIER_MATCH",
        col_nombre_b: "NOMBRE_B_MATCH",
        "SCORE_SIM": "SCORE_SIM_MATCH",
        "SCORE_100": "SCORE_100_MATCH",
    }
    cols_match = [
        c
        for c in ["ID_A", "ID_B", col_nombre_b, "TIER", "SCORE_SIM", "SCORE_100"]
        if c in mejor.columns
    ]
    if "FLAG_GUARDRAIL" in mejor.columns:
        cols_match.append("FLAG_GUARDRAIL")
    out = base.merge(
        mejor[cols_match].rename(columns=ren), left_on="_ID", right_on="ID_A", how="left"
    ).drop(columns="ID_A", errors="ignore")
    out["TIENE_MATCH"] = out["TIER_MATCH"].notna()
    out["TIER_MATCH"] = out["TIER_MATCH"].fillna("NO_MATCH")
    return out


def construir_control_calidad(
    df_a: pd.DataFrame,
    df_b: pd.DataFrame,
    comb: pd.DataFrame,
    correlativa: pd.DataFrame,
    base_a_match: pd.DataFrame,
    config: ConfigCruce,
    meta: dict[str, int],
) -> pd.DataFrame:
    """Controles de calidad del cruce (una fila por control)."""
    filas: list[dict[str, Any]] = []

    def add(control: str, valor: Any, estado: str = "OK", detalle: str = "") -> None:
        filas.append({"control": control, "valor": valor, "estado": estado, "detalle": detalle})

    add("filas_base_A", len(df_a))
    add("filas_base_B", len(df_b))
    add("filas_A_en_matching", meta["filas_a"])
    add("filas_B_en_matching", meta["filas_b"])
    add(
        "descartadas_por_clave_vacia",
        meta["descartadas_clave_vacia"],
        "REVISAR" if meta["descartadas_clave_vacia"] else "OK",
    )
    add("pares_emparejados_total", len(correlativa))
    if not correlativa.empty:
        for tier, n in correlativa["TIER"].value_counts().items():
            add(f"pares_{tier}", int(n))
        empresas_a_con_match = int(base_a_match["TIENE_MATCH"].sum())
        add("empresas_A_con_match", empresas_a_con_match)
        add("empresas_A_sin_match", int(len(df_a) - empresas_a_con_match))
        multi = correlativa.groupby("ID_A")["ID_B"].nunique()
        n_multi = int((multi > 1).sum())
        add(
            "empresas_A_con_multiples_matches",
            n_multi,
            "REVISAR" if n_multi else "OK",
            "Una empresa A empareja con >1 registro B: revisar duplicados o ambigüedad.",
        )
        for v in config.variables_guardrail:
            col = f"{v.nombre}_COMPATIBLE"
            if col in correlativa.columns:
                incompat = int(correlativa[col].eq(False).sum())
                add(
                    f"pares_{v.nombre}_incompatible",
                    incompat,
                    "REVISAR" if incompat else "OK",
                    f"Match de nombre con {v.nombre} distinto: verificar antes de usar.",
                )
    return pd.DataFrame(filas)


def construir_resumen(
    config: ConfigCruce,
    espec_a: EspecBase,
    espec_b: EspecBase,
    df_a: pd.DataFrame,
    df_b: pd.DataFrame,
    correlativa: pd.DataFrame,
    base_a_match: pd.DataFrame,
) -> pd.DataFrame:
    """Resumen ejecutivo del cruce (una métrica por fila)."""
    vars_desc = "; ".join(f"{v.nombre}[{v.rol}:{v.comparador}]" for v in config.variables)
    filas: list[tuple[str, Any]] = [
        ("cruce", config.nombre),
        ("base_A", f"{espec_a.nombre} · {espec_a.archivo} · hoja='{espec_a.hoja}'"),
        ("base_B", f"{espec_b.nombre} · {espec_b.archivo} · hoja='{espec_b.hoja}'"),
        ("variables_de_union", vars_desc),
        ("umbral_seguro", config.umbral_seguro),
        ("umbral_revision", config.umbral_revision),
        ("idf_weight", config.idf_weight),
        ("modo_limpieza_nombres", config.modo_limpieza_nombres),
        ("filas_base_A", len(df_a)),
        ("filas_base_B", len(df_b)),
        ("pares_emparejados_total", len(correlativa)),
    ]
    if not correlativa.empty:
        for tier, n in correlativa["TIER"].value_counts().items():
            filas.append((f"pares_{tier}", int(n)))
        filas.append(("empresas_A_con_match", int(base_a_match["TIENE_MATCH"].sum())))
    filas.append(
        (
            "advertencia_calidad",
            "Sin ground truth etiquetado no se garantiza F1; se prioriza precisión y "
            "auditabilidad sobre recall. Revise SIEMPRE el nivel REVISIÓN.",
        )
    )
    return pd.DataFrame(filas, columns=["metrica", "valor"])


def guardar_metadata(rutas: RutasCruce, config: ConfigCruce, stats: dict[str, Any]) -> None:
    """Guarda metadata JSON que liga *qué* se produjo con *cómo* (reproducibilidad)."""
    meta = {
        "timestamp": datetime.now().isoformat(),
        "version_cruce_core": __version__,
        "cruce": config.nombre,
        "config": {
            "umbral_seguro": config.umbral_seguro,
            "umbral_revision": config.umbral_revision,
            "idf_weight": config.idf_weight,
            "min_concordancias": config.min_concordancias,
            "modo_limpieza_nombres": config.modo_limpieza_nombres,
            "max_bucket_size": config.max_bucket_size,
            "variables": [
                {
                    "nombre": v.nombre,
                    "col_a": v.col_a,
                    "col_b": v.col_b,
                    "comparador": v.comparador,
                    "rol": v.rol,
                    "umbral_concordancia": v.umbral_concordancia,
                    "peso": v.peso,
                }
                for v in config.variables
            ],
        },
        "stats": stats,
        "random_seed": 42,
    }
    (rutas.dir_salida / f"metadata_{config.nombre}.json").write_text(
        json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print("📋 Metadata guardada.")


def _asegurar_columnas_unicas(df: pd.DataFrame) -> pd.DataFrame:
    """Renombra columnas duplicadas con sufijo .1, .2… (Excel/Parquet no admiten dups)."""
    if not df.columns.duplicated().any():
        return df
    nuevos: list[str] = []
    vistos: dict[str, int] = {}
    for c in df.columns:
        if c in vistos:
            vistos[c] += 1
            nuevos.append(f"{c}.{vistos[c]}")
        else:
            vistos[c] = 0
            nuevos.append(c)
    out = df.copy()
    out.columns = nuevos
    return out


def exportar(
    rutas: RutasCruce,
    config: ConfigCruce,
    correlativa: pd.DataFrame,
    base_a_match: pd.DataFrame,
    control: pd.DataFrame,
    resumen: pd.DataFrame,
    formatos: tuple[str, ...] = ("csv", "xlsx"),
) -> list[Path]:
    """Exporta los resultados del cruce en los formatos pedidos."""
    rutas.crear_directorios()
    n = config.nombre
    generados: list[Path] = []

    seguros = (
        correlativa[correlativa["TIER"] == "MATCH_SEGURO"] if not correlativa.empty else correlativa
    )
    revision = (
        correlativa[correlativa["TIER"] != "MATCH_SEGURO"] if not correlativa.empty else correlativa
    )

    artefactos = {
        f"correlativa_{n}": correlativa,
        f"matches_seguros_{n}": seguros,
        f"matches_revision_{n}": revision,
        f"baseA_con_match_{n}": base_a_match,
        f"control_calidad_{n}": control,
        f"resumen_{n}": resumen,
    }
    artefactos = {k: _asegurar_columnas_unicas(v) for k, v in artefactos.items()}
    if "csv" in formatos:
        for nombre, df in artefactos.items():
            p = rutas.dir_salida / f"{nombre}.csv"
            df.to_csv(p, index=False, encoding="utf-8-sig")
            generados.append(p)
    if "parquet" in formatos:
        for nombre, df in artefactos.items():
            p = rutas.dir_salida / f"{nombre}.parquet"
            df.astype("string").to_parquet(p, index=False)
            generados.append(p)
    if "xlsx" in formatos:
        p = rutas.dir_salida / f"cruce_{n}_resultados.xlsx"
        _exportar_excel(p, resumen, correlativa, seguros, revision, base_a_match, control)
        generados.append(p)

    print(f"📦 Exportados {len(generados)} archivos en {rutas.dir_salida}")
    return generados


def _exportar_excel(
    ruta: Path,
    resumen: pd.DataFrame,
    correlativa: pd.DataFrame,
    seguros: pd.DataFrame,
    revision: pd.DataFrame,
    base_a_match: pd.DataFrame,
    control: pd.DataFrame,
) -> None:
    """Escribe el Excel consolidado con formato (xlsxwriter)."""
    hojas = {
        "00_resumen": resumen,
        "01_correlativa": correlativa,
        "02_matches_seguros": seguros,
        "03_revision_manual": revision,
        "04_baseA_con_match": base_a_match,
        "05_control_calidad": control,
    }
    with pd.ExcelWriter(ruta, engine="xlsxwriter") as writer:
        wb = writer.book
        header_fmt = wb.add_format({"bold": True, "bg_color": "#D9EAF7", "border": 1})
        ok_fmt = wb.add_format({"bg_color": "#E2F0D9"})
        warn_fmt = wb.add_format({"bg_color": "#FFF2CC"})
        for nombre, df in hojas.items():
            df = _asegurar_columnas_unicas(df)
            df.to_excel(writer, sheet_name=nombre, index=False)
            ws = writer.sheets[nombre]
            ws.freeze_panes(1, 0)
            if len(df.columns):
                ws.autofilter(0, 0, max(len(df), 1), len(df.columns) - 1)
            for j, col in enumerate(df.columns):
                ws.write(0, j, str(col), header_fmt)
                muestra = df.iloc[:, j].head(200).tolist() if len(df) else []
                ancho = min(max([len(str(col))] + [len(str(x)) for x in muestra] + [6]) + 2, 48)
                ws.set_column(j, j, ancho)
            if "TIER" in df.columns and len(df):
                c = df.columns.get_loc("TIER")
                ws.conditional_format(
                    1,
                    c,
                    len(df),
                    c,
                    {"type": "text", "criteria": "containing", "value": "SEGURO", "format": ok_fmt},
                )
                ws.conditional_format(
                    1,
                    c,
                    len(df),
                    c,
                    {
                        "type": "text",
                        "criteria": "containing",
                        "value": "REVISION",
                        "format": warn_fmt,
                    },
                )


# ═══════════════════════════════════════════════════════════════════════════
# 9. OBSERVABILIDAD
# ═══════════════════════════════════════════════════════════════════════════
def reporte_composicion(correlativa: pd.DataFrame, df_a: pd.DataFrame) -> None:
    """Reporte visual de composición del cruce (barras por nivel)."""
    total_a = len(df_a)
    print(f"\n{'=' * 56}\n📊 COMPOSICIÓN DEL CRUCE\n{'=' * 56}")
    print(f"  Empresas en Base A : {total_a:>8,}")
    print(f"  Pares emparejados  : {len(correlativa):>8,}")
    if not correlativa.empty:
        for tier, n in correlativa["TIER"].value_counts().items():
            barra = "█" * int(n / max(1, len(correlativa)) * 22)
            print(f"  {tier!s:<28} {n:>7,}  {n / len(correlativa):>6.1%}  {barra}")
    print("=" * 56)


# ═══════════════════════════════════════════════════════════════════════════
# 10. ORQUESTADORES
# ═══════════════════════════════════════════════════════════════════════════
def ejecutar_cruce(
    config: ConfigCruce,
    espec_a: EspecBase,
    espec_b: EspecBase,
    df_a: pd.DataFrame,
    df_b: pd.DataFrame,
    rutas: RutasCruce,
    canon_categoricos: dict[str, str],
    formatos: tuple[str, ...] = ("csv", "xlsx"),
    exportar_resultados: bool = True,
) -> dict[str, Any]:
    """Ejecuta un cruce completo (preparar → bloquear → puntuar → clasificar → exportar)."""
    t0 = time.time()
    print(f"\n{'━' * 60}\n▶ CRUCE: {config.nombre}\n{'━' * 60}")
    comb, meta = preparar_combinado(df_a, df_b, espec_a, espec_b, config, canon_categoricos)
    pares = generar_pares_cruzados(comb, config, espec_a.nombre)
    matches = puntuar_y_clasificar(comb, pares, config)
    correlativa = construir_correlativa(matches, comb, config)
    base_a_match = construir_base_a_con_match(df_a, correlativa, config)
    control = construir_control_calidad(df_a, df_b, comb, correlativa, base_a_match, config, meta)
    resumen = construir_resumen(config, espec_a, espec_b, df_a, df_b, correlativa, base_a_match)

    stats = {
        "candidatos_cruzados": len(pares),
        "pares_emparejados": len(correlativa),
        "por_nivel": (
            correlativa["TIER"].value_counts().to_dict() if not correlativa.empty else {}
        ),
        "empresas_A_con_match": int(base_a_match["TIENE_MATCH"].sum()),
        "segundos": round(time.time() - t0, 2),
        **meta,
    }
    if exportar_resultados:
        exportar(rutas, config, correlativa, base_a_match, control, resumen, formatos)
        guardar_metadata(rutas, config, stats)
    reporte_composicion(correlativa, df_a)
    print(f"⏱️  Cruce '{config.nombre}' completado en {stats['segundos']}s.")
    return {
        "correlativa": correlativa,
        "base_a_match": base_a_match,
        "control": control,
        "resumen": resumen,
        "stats": stats,
    }


def ejecutar_todos(
    cruces: dict[str, ConfigCruce],
    espec_a: EspecBase,
    espec_b: EspecBase,
    dir_datos: str | Path,
    canon_categoricos: dict[str, str] | None = None,
    formatos: tuple[str, ...] = ("csv", "xlsx"),
    smoke_test: bool = True,
    n_smoke: int = 200,
) -> dict[str, dict[str, Any]]:
    """Ejecuta todos los cruces activos: pre-flight → carga → (smoke) → cruces."""
    canon = CANON_PAISES_DEFAULT if canon_categoricos is None else canon_categoricos
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    activos = {n: c for n, c in cruces.items() if c.activo}

    rutas_pf = construir_rutas(dir_datos, espec_a, espec_b, "preflight", ts)
    preflight(espec_a, espec_b, rutas_pf, cruces)

    df_a = cargar_base(espec_a, rutas_pf.archivo_a)
    df_b = cargar_base(espec_b, rutas_pf.archivo_b)

    if smoke_test:
        print(f"\n🔬 SMOKE TEST (muestra de {n_smoke} por base)…")
        sa = df_a.sample(n=min(n_smoke, len(df_a)), random_state=42).reset_index(drop=True)
        sb = df_b.sample(n=min(n_smoke, len(df_b)), random_state=42).reset_index(drop=True)
        for nombre, cfg in activos.items():
            rs = construir_rutas(dir_datos, espec_a, espec_b, f"smoke_{nombre}", ts)
            ejecutar_cruce(
                cfg, espec_a, espec_b, sa, sb, rs, canon, formatos=(), exportar_resultados=False
            )
        print("✅ Smoke test OK — sintaxis, rutas y tipos validados.\n")

    resultados: dict[str, dict[str, Any]] = {}
    for nombre, cfg in activos.items():
        rutas = construir_rutas(dir_datos, espec_a, espec_b, nombre, ts)
        resultados[nombre] = ejecutar_cruce(
            cfg, espec_a, espec_b, df_a, df_b, rutas, canon, formatos
        )
    return resultados

print("✅ Celda EXTRAS lista: lógica de cruce (v2, IDF) cargada. Edite y ejecute la Celda B.")


✅ Celda EXTRAS lista: lógica de cruce (v2, IDF) cargada. Edite y ejecute la Celda B.


## Celda B — EJECUTAR  ·  *la única celda que usted edita*

Declare rutas, las dos bases y los cruces. Al ejecutarla se monta Drive, se instala el paquete y se corren todos los cruces activos.

In [3]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — CONFIGURACIÓN Y EJECUCIÓN        (LA ÚNICA CELDA QUE USTED EDITA)
# ══════════════════════════════════════════════════════════════════════════

# 1) RUTAS EN GOOGLE DRIVE ──────────────────────────────────────────────────
RUTA_PAQUETE = "/content/drive/MyDrive/ProColombia/record_linkage"
DIR_DATOS = "/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-07-08"

# 2) BASE A  (queda a la IZQUIERDA del cruce) ───────────────────────────────
BASE_A = EspecBase(
    nombre="GAZELLE_NEARSHORING",
    archivo="2021-04-19 Base Segmentación Gazelle - Nearshoring (002).xlsx",
    hoja="Base",
    fila_encabezado=0,  # encabezado en la primera fila
    prefijo_id="A",
    columnas_extra=("Main Email", "Primary Naics", "País"),  # se arrastran a la correlativa
)

# 3) BASE B  (queda a la DERECHA del cruce) ─────────────────────────────────
BASE_B = EspecBase(
    nombre="EMPRESAS_SERVICIOS",
    archivo="2026-07-08 Empresas con servicios_(2022-Jun2026).xlsx",
    hoja="Consolidado",
    fila_encabezado=8,  # OJO: el encabezado real está en la fila 9 de Excel (índice 8)
    prefijo_id="B",
    columnas_extra=("Tipo de Cuenta", "Cuenta ID"),
)

# 4) MAPA CANÓNICO DE PAÍSES  (alinea "Estados Unidos de América" ↔ "Estados Unidos")
CANON_PAISES = dict(CANON_PAISES_DEFAULT)  # extienda con {"FORMA_ORIGEN": "FORMA_CANONICA", ...}

# 5) CRUCES A EJECUTAR ──────────────────────────────────────────────────────
#    Comparador recomendado para razón social: "nombre_idf". Pondera los tokens
#    por su rareza (IDF): las palabras genéricas (CORPORATION, GROUP,
#    INTERNATIONAL, TECHNOLOGIES…) pesan casi nada y el token distintivo manda.
#    Esto evita falsos positivos como "ACETO CORPORATION" ↔ "AGCO CORPORATION".
CRUCES = {
    # ── Cruce 1: SOLO por nombre de empresa ────────────────────────────────
    "01_solo_nombre": ConfigCruce(
        nombre="01_solo_nombre",
        variables=(
            EspecVariable(
                "RAZON_SOCIAL", "Empresa", "Nombre Cuenta",
                comparador="nombre_idf", rol="clave",
            ),
        ),
        umbral_seguro=0.90,
        umbral_revision=0.80,
        idf_weight=0.6,  # 0..1: peso del coseno IDF frente a Jaro-Winkler
    ),
    # ── Cruce 2: nombre + país (país como guardrail) ───────────────────────
    "02_nombre_pais": ConfigCruce(
        nombre="02_nombre_pais",
        variables=(
            EspecVariable(
                "RAZON_SOCIAL", "Empresa", "Nombre Cuenta",
                comparador="nombre_idf", rol="clave",
            ),
            EspecVariable(
                "PAIS", "País", "País Inversionista",
                comparador="categorico", rol="guardrail",
            ),
        ),
        umbral_seguro=0.90,
        umbral_revision=0.80,
        idf_weight=0.6,
    ),
    # ── PLANTILLA: copie, renombre y ajuste para un nuevo cruce ────────────
    # "03_mi_cruce": ConfigCruce(
    #     nombre="03_mi_cruce",
    #     variables=(
    #         EspecVariable("RAZON_SOCIAL", "Empresa", "Nombre Cuenta",
    #                       comparador="nombre_idf", rol="clave"),
    #         EspecVariable("PAIS", "País", "País Inversionista",
    #                       comparador="categorico", rol="guardrail"),
    #         # Evidencia adicional (rol="score") o prohibición (rol="veto"):
    #         # EspecVariable("NIT", "Nit", "NIT", comparador="nit", rol="veto"),
    #     ),
    #     umbral_seguro=0.90, umbral_revision=0.80, idf_weight=0.6,
    #     activo=True,      # ponga False para saltar este cruce sin borrarlo
    # ),
}

# 6) FORMATOS DE SALIDA ──────────────────────────────────────────────────────
FORMATOS = ("csv", "xlsx")  # agregue "parquet" para re-cargar rápido en Python

# ── EJECUCIÓN (no requiere edición) ──────────────────────────────────────────
montar_drive_con_retry()
instalar_paquete(RUTA_PAQUETE)
resultados = ejecutar_todos(
    cruces=CRUCES,
    espec_a=BASE_A,
    espec_b=BASE_B,
    dir_datos=DIR_DATOS,
    canon_categoricos=CANON_PAISES,
    formatos=FORMATOS,
    smoke_test=True,  # corre una muestra antes del run completo (recomendado)
    n_smoke=200,
)
print("\n✅ Listo. Las salidas quedaron en:  <DIR_DATOS>/cruces/<nombre_cruce>_<timestamp>/")


Mounted at /content/drive
✅ Google Drive montado.
⏳ Instalando rues-linker en modo editable…
⏳ Reintentando con --force-reinstall…
✅ record_linkage disponible (versión 0.7.5).
✅ Pre-flight OK — 2 cruce(s) activo(s); columnas verificadas.
✅ GAZELLE_NEARSHORING: 1,714 filas × 72 columnas — 2021-04-19 Base Segmentación Gazelle - Nearshoring (002).xlsx
✅ EMPRESAS_SERVICIOS: 10,150 filas × 4 columnas — 2026-07-08 Empresas con servicios_(2022-Jun2026).xlsx

🔬 SMOKE TEST (muestra de 200 por base)…

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ CRUCE: 01_solo_nombre
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026-07-09 14:12:53 | root                      | INFO     | Sistema de Logging configurado exitosamente. Usando hora de Colombia.
✅ Combinado: 400 filas (A=200 · B=200) — 0 descartadas por 'RAZON_SOCIAL' vacío.
2026-07-09 14:12:54 | record_linkage.engine.lsh.name_blocking | INFO     | [name_blocking/fingerprint] 0 buckets emitidos (0 descartados) → +0 pa

## Celda C — Inspección de resultados  ·  *opcional*

In [4]:
# ── Inspección rápida de resultados EN SESIÓN (opcional) ─────────────────────
# Muestra los mejores matches de cada cruce sin abrir los archivos de Drive.
for nombre_cruce, r in resultados.items():
    corr = r["correlativa"]
    print(f"\n═══ {nombre_cruce}: {len(corr)} pares · {r['stats']['por_nivel']} ═══")
    if not corr.empty:
        columnas = [
            c
            for c in [
                "ID_A", "ID_B", "TIER", "SCORE_100",
                "RAZON_SOCIAL_A", "RAZON_SOCIAL_B",
                "PAIS_A", "PAIS_B", "PAIS_COMPATIBLE", "FLAG_GUARDRAIL",
            ]
            if c in corr.columns
        ]
        display(corr[columnas].head(15))  # noqa: F821  (display existe en Colab)



═══ 01_solo_nombre: 47 pares · {'MATCH_SEGURO': 32, 'MATCH_REVISION': 15} ═══


,ID_A,ID_B,TIER,SCORE_100,RAZON_SOCIAL_A,RAZON_SOCIAL_B
0,A_000762,B_001227,MATCH_SEGURO,100.0,BLACK & VEATCH,Black & Veatch
1,A_000433,B_008141,MATCH_SEGURO,100.0,RIPPLE,Ripple
2,A_000391,B_005842,MATCH_SEGURO,100.0,AIR PRODUCTS,Air Products
3,A_000297,B_004572,MATCH_SEGURO,100.0,TELETECH,TELETECH
4,A_000265,B_000942,MATCH_SEGURO,100.0,COLUMBIA UNIVERSITY,Columbia University
5,A_000689,B_000770,MATCH_SEGURO,100.0,DUKE UNIVERSITY,Duke university
6,A_000724,B_003452,MATCH_SEGURO,100.0,CUSHMAN WAKEFIELD,Cushman & Wakefield
7,A_001150,B_003622,MATCH_SEGURO,100.0,EDSA,EDSA
8,A_000712,B_006281,MATCH_SEGURO,100.0,SILICON VALLEY BANK,Silicon Valley Bank
9,A_000724,B_004795,MATCH_SEGURO,100.0,CUSHMAN WAKEFIELD,Cushman & Wakefield



═══ 02_nombre_pais: 47 pares · {'MATCH_SEGURO': 29, 'MATCH_REVISION': 15, 'MATCH_REVISION_GUARDRAIL': 3} ═══


,ID_A,ID_B,TIER,SCORE_100,RAZON_SOCIAL_A,RAZON_SOCIAL_B,PAIS_A,PAIS_B,PAIS_COMPATIBLE,FLAG_GUARDRAIL
0,A_000762,B_001227,MATCH_SEGURO,100.0,BLACK & VEATCH,Black & Veatch,<NA>,Estados Unidos,None,
1,A_000433,B_008141,MATCH_SEGURO,100.0,RIPPLE,Ripple,<NA>,Estados Unidos,None,
2,A_000391,B_005842,MATCH_SEGURO,100.0,AIR PRODUCTS,Air Products,<NA>,Estados Unidos,None,
3,A_000297,B_004572,MATCH_SEGURO,100.0,TELETECH,TELETECH,<NA>,Estados Unidos,None,
4,A_000265,B_000942,MATCH_SEGURO,100.0,COLUMBIA UNIVERSITY,Columbia University,<NA>,Estados Unidos,None,
5,A_000689,B_000770,MATCH_SEGURO,100.0,DUKE UNIVERSITY,Duke university,Estados Unidos de América,Estados Unidos,True,
6,A_000724,B_003452,MATCH_SEGURO,100.0,CUSHMAN WAKEFIELD,Cushman & Wakefield,Estados Unidos de América,Estados Unidos,True,
7,A_000712,B_006281,MATCH_SEGURO,100.0,SILICON VALLEY BANK,Silicon Valley Bank,Estados Unidos de América,Estados Unidos,True,
8,A_000724,B_004795,MATCH_SEGURO,100.0,CUSHMAN WAKEFIELD,Cushman & Wakefield,Estados Unidos de América,Estados Unidos,True,
9,A_000930,B_007599,MATCH_SEGURO,100.0,THE WORLD BANK,THE WORLD BANK,<NA>,Estados Unidos,None,


## Celda D — Resumen y cierre  ·  *opcional*

In [5]:
# ── Resumen de sesión ────────────────────────────────────────────────────────
for nombre_cruce, r in resultados.items():
    s = r["stats"]
    print(
        f"{nombre_cruce}: {s['pares_emparejados']} pares en {s['segundos']}s  →  {s['por_nivel']}"
    )

# ── Liberar el runtime al terminar (DESCOMENTE si desea apagar la sesión) ─────
# from google.colab import runtime
# runtime.unassign()


01_solo_nombre: 47 pares en 0.8s  →  {'MATCH_SEGURO': 32, 'MATCH_REVISION': 15}
02_nombre_pais: 47 pares en 0.73s  →  {'MATCH_SEGURO': 29, 'MATCH_REVISION': 15, 'MATCH_REVISION_GUARDRAIL': 3}
